In [ ]:
!pip install datasets pandas scikit-learn

import pandas as pd
import random
from datasets import load_dataset

In [ ]:

dataset = load_dataset("snips_built_in_intents")

df = pd.DataFrame(dataset['train'])

df = df[['text', 'label']]
df = df.rename(columns={'label': 'intent'})

intent_names = dataset['train'].features['label'].names

df['intent'] = df['intent'].apply(lambda x: intent_names[x])

print(df.head())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


                                      text                intent
0  Share my location with Hillary's sister  ShareCurrentLocation
1    Send my current location to my father  ShareCurrentLocation
2       Share my current location with Jim  ShareCurrentLocation
3           Send my location to my husband  ShareCurrentLocation
4                         Send my location  ShareCurrentLocation


In [ ]:
print(df.columns)

Index(['text', 'intent'], dtype='object')


In [ ]:
selected_intents = df['intent'].unique()[:10]
df = df[df['intent'].isin(selected_intents)].reset_index(drop=True)

In [ ]:
print(df['intent'].unique())

['ShareCurrentLocation' 'ComparePlaces' 'GetPlaceDetails' 'SearchPlace'
 'BookRestaurant' 'RequestRide' 'GetDirections' 'ShareETA'
 'GetTrafficInformation' 'GetWeather']


In [ ]:
intent_mapping = {
    "ShareCurrentLocation": "aadhaar_update",
    "ComparePlaces": "passport_apply",
    "GetPlaceDetails": "passport_status",
    "SearchPlace": "general_query",
    "BookRestaurant": "pan_apply",
    "RequestRide": "dl_apply",
    "GetDirections": "dl_status",
    "ShareETA": "aadhaar_status",
    "GetTrafficInformation": "dl_renewal",
    "GetWeather": "pan_status"
}

In [ ]:
df['intent'] = df['intent'].map(intent_mapping)
df = df.dropna().reset_index(drop=True)

In [ ]:
print(df['intent'].unique())
print("Dataset size:", len(df))
print(df.head())

['aadhaar_update' 'passport_apply' 'passport_status' 'general_query'
 'pan_apply' 'dl_apply' 'dl_status' 'aadhaar_status' 'dl_renewal'
 'pan_status']
Dataset size: 328
                                      text          intent
0  Share my location with Hillary's sister  aadhaar_update
1    Send my current location to my father  aadhaar_update
2       Share my current location with Jim  aadhaar_update
3           Send my location to my husband  aadhaar_update
4                         Send my location  aadhaar_update


In [ ]:
keyword_map = {
    "aadhaar_update": "aadhaar update",
    "aadhaar_status": "aadhaar status",
    "pan_status": "pan status",
    "pan_apply": "pan apply",
    "passport_apply": "passport apply",
    "passport_status": "passport status",
    "dl_apply": "driving license apply",
    "dl_status": "driving license status",
    "dl_renewal": "dl renewal",
    "general_query": "government service"
}

In [ ]:
def convert_to_gov(text, intent):
    keyword = keyword_map[intent]

    # simple transformation
    return f"{keyword} {text.lower()}"

In [ ]:
df['text'] = df.apply(lambda x: convert_to_gov(x['text'], x['intent']), axis=1)

In [ ]:
print(df.head())

                                                text          intent
0  aadhaar update share my location with hillary'...  aadhaar_update
1  aadhaar update send my current location to my ...  aadhaar_update
2  aadhaar update share my current location with jim  aadhaar_update
3      aadhaar update send my location to my husband  aadhaar_update
4                    aadhaar update send my location  aadhaar_update


In [ ]:
def convert_to_gov(text, intent):
    keyword = keyword_map[intent]

    formats = [
        f"{keyword} {text}",
        f"{text} for {keyword}",
        f"how to {keyword}",
        f"{keyword} kaise kare"
    ]

    return random.choice(formats)

In [ ]:
print(df.head())

                                                text          intent
0  aadhaar update share my location with hillary'...  aadhaar_update
1  aadhaar update send my current location to my ...  aadhaar_update
2  aadhaar update share my current location with jim  aadhaar_update
3      aadhaar update send my location to my husband  aadhaar_update
4                    aadhaar update send my location  aadhaar_update


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# split
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['intent'], test_size=0.2, random_state=42
)

# vectorizer
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

# evaluate
y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.9848484848484849


In [ ]:
import random

y_test_shuffled = y_test.sample(frac=1)

print("Sanity Check Accuracy:",
      accuracy_score(y_test_shuffled, y_pred))

Sanity Check Accuracy: 0.22727272727272727


In [ ]:
import random

def add_noise(text):
    text = list(text)
    if len(text) > 5:
        i = random.randint(0, len(text)-1)
        text.pop(i)
    return "".join(text)

def to_hinglish(text):
    return text.replace("how", "kaise").replace("check", "check karna")

In [ ]:
complex_df = df.copy()

complex_df['text'] = complex_df['text'].apply(to_hinglish)
complex_df['text'] = complex_df['text'].apply(add_noise)

In [ ]:
X_complex = vectorizer.transform(complex_df['text'])

y_complex_pred = model.predict(X_complex)

from sklearn.metrics import accuracy_score

print("Complex Accuracy:",
      accuracy_score(complex_df['intent'], y_complex_pred))

Complex Accuracy: 0.9939024390243902


In [ ]:
complex_templates = {
    "aadhaar_update": [
        "mera id galat hai kaise theek karu",
        "details change karni hai kya karu",
        "aadhaar me correction kaise hota hai"
    ],

    "pan_status": [
        "pan bana ki nahi kaise pata chale",
        "mera application ka status kya hai",
        "status check karna hai"
    ],

    "passport_apply": [
        "passport banwana hai process kya hai",
        "new passport ke liye apply kaise kare",
        "passport kaise banega"
    ],

    "dl_apply": [
        "dl banwana hai kaise kare",
        "license apply karna hai",
        "driving license ka process kya hai"
    ],

    "dl_status": [
        "mera license bana ya nahi",
        "dl ka status kya hai",
        "license approve hua kya"
    ],

    "aadhaar_status": [
        "mera id update hua ya nahi",
        "aadhaar ka status kya hai",
        "check karna hai update hua kya"
    ],

    "pan_apply": [
        "pan card banana hai",
        "new pan ke liye apply kaise kare",
        "pan banwana hai"
    ],

    "passport_status": [
        "passport bana ya nahi",
        "passport ka status kya hai",
        "track passport request"
    ],

    "dl_renewal": [
        "license renew karna hai",
        "dl expire ho gaya renew kaise kare",
        "renewal ka process kya hai"
    ],

    "general_query": [
        "mujhe help chahiye",
        "koi help karo",
        "government service ka info chahiye"
    ]
}

In [ ]:
import random

complex_data = []

for intent in df['intent'].unique():
    for _ in range(50):  # 50 samples per intent
        text = random.choice(complex_templates[intent])
        complex_data.append((text, intent))

complex_df = pd.DataFrame(complex_data, columns=['text','intent'])

In [ ]:
svm_ref = model            # when model was LogisticRegression

In [ ]:
X_complex = vectorizer.transform(complex_df['text'])
y_complex_pred = model.predict(X_complex)

from sklearn.metrics import accuracy_score
print("Complex Accuracy:", accuracy_score(complex_df['intent'], y_complex_pred))

Complex Accuracy: 0.436


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['intent'], test_size=0.2, random_state=42
)

In [ ]:
labels = list(df['intent'].unique())

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

y_train_enc = y_train.map(label2id)
y_test_enc = y_test.map(label2id)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(X_train), truncation=True, padding=True)
test_encodings = tokenizer(list(X_test), truncation=True, padding=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch

class DatasetTorch(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = DatasetTorch(train_encodings, y_train_enc)
test_dataset = DatasetTorch(test_encodings, y_test_enc)

In [ ]:
bert_ref = model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(labels)
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=99, training_loss=1.064414669768979, metrics={'train_runtime': 186.4402, 'train_samples_per_second': 4.216, 'train_steps_per_second': 0.531, 'total_flos': 10098639847800.0, 'train_loss': 1.064414669768979, 'epoch': 3.0})

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

print("BERT Clean Accuracy:",
      accuracy_score(y_test_enc, y_pred))

BERT Clean Accuracy: 1.0


In [ ]:
# encode labels
complex_df['label'] = complex_df['intent'].map(label2id)

# tokenize
complex_encodings = tokenizer(list(complex_df['text']), truncation=True, padding=True)

complex_dataset = DatasetTorch(complex_encodings, complex_df['label'])

# predict
pred_complex = trainer.predict(complex_dataset)

y_complex_pred = np.argmax(pred_complex.predictions, axis=1)

print("BERT Complex Accuracy:",
      accuracy_score(complex_df['label'], y_complex_pred))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


BERT Complex Accuracy: 0.412


In [ ]:
for i in range(20):
    if complex_df['label'].iloc[i] != y_complex_pred[i]:
        print("TEXT:", complex_df['text'].iloc[i])
        print("ACTUAL:", complex_df['label'].iloc[i])
        print("PREDICTED:", y_complex_pred[i])
        print("------")

TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: aadhaar me correction kaise hota hai
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: aadhaar me correction kaise hota hai
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: aadhaar me correction kaise hota hai
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: 0
PREDICTED: 7
------


In [ ]:
predicted_labels = [id2label[i] for i in y_complex_pred]
actual_labels = complex_df['intent']

In [ ]:
for i in range(20):
    if actual_labels.iloc[i] != predicted_labels[i]:
        print("TEXT:", complex_df['text'].iloc[i])
        print("ACTUAL:", actual_labels.iloc[i])
        print("PREDICTED:", predicted_labels[i])
        print("------")

TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: aadhaar me correction kaise hota hai
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: aadhaar me correction kaise hota hai
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: aadhaar me correction kaise hota hai
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai kaise theek karu
ACTUAL: aadhaar_update
PREDICTED: aadhaar_status
------
TEXT: mera id galat hai 

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer_dl = Tokenizer(num_words=5000)
tokenizer_dl.fit_on_texts(X_train)

X_train_seq = tokenizer_dl.texts_to_sequences(X_train)
X_test_seq = tokenizer_dl.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=20)
X_test_pad = pad_sequences(X_test_seq, maxlen=20)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_dl = le.fit_transform(y_train)
y_test_dl = le.transform(y_test)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense

model_dl = Sequential([
    Embedding(input_dim=5000, output_dim=64),
    Bidirectional(LSTM(64)),
    Dense(64, activation='relu'),
    Dense(len(le.classes_), activation='softmax')  # now works
])

In [ ]:
model_dl.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model_dl.fit(
    X_train_pad, y_train_dl,
    epochs=5,
    batch_size=16,
    validation_data=(X_test_pad, y_test_dl)
)

Epoch 1/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - accuracy: 0.2137 - loss: 2.2851 - val_accuracy: 0.4091 - val_loss: 2.2240
Epoch 2/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3397 - loss: 2.1195 - val_accuracy: 0.4394 - val_loss: 1.6758
Epoch 3/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3702 - loss: 1.6580 - val_accuracy: 0.4848 - val_loss: 1.4072
Epoch 4/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4924 - loss: 1.2611 - val_accuracy: 0.7879 - val_loss: 0.9201
Epoch 5/5
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7824 - loss: 0.7546 - val_accuracy: 0.8182 - val_loss: 0.5816


In [ ]:
loss, acc = model_dl.evaluate(X_test_pad, y_test_dl)
print("BiLSTM Clean Accuracy:", acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8182 - loss: 0.5816
BiLSTM Clean Accuracy: 0.8181818127632141


In [ ]:
X_complex_seq = tokenizer_dl.texts_to_sequences(complex_df['text'])
X_complex_pad = pad_sequences(X_complex_seq, maxlen=20)

y_complex_dl = le.transform(complex_df['intent'])

loss, acc = model_dl.evaluate(X_complex_pad, y_complex_dl)
print("BiLSTM Complex Accuracy:", acc)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1380 - loss: 3.3172
BiLSTM Complex Accuracy: 0.1379999965429306


In [ ]:
X_train_pad = pad_sequences(X_train_seq, maxlen=40)
X_test_pad = pad_sequences(X_test_seq, maxlen=40)

In [ ]:
model_dl = Sequential([
    Embedding(input_dim=10000, output_dim=128),
    Bidirectional(LSTM(128)),
    Dense(64, activation='relu'),
    Dense(len(le.classes_), activation='softmax')
])

In [ ]:
model_dl.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
model_dl.fit(
    X_train_pad, y_train_dl,
    epochs=10,
    batch_size=16,
    validation_data=(X_test_pad, y_test_dl)
)

Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 80ms/step - accuracy: 0.1603 - loss: 2.2478 - val_accuracy: 0.2576 - val_loss: 2.1337
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.2672 - loss: 2.0875 - val_accuracy: 0.4545 - val_loss: 1.7575
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.4084 - loss: 1.6739 - val_accuracy: 0.5455 - val_loss: 1.4312
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.6298 - loss: 1.2147 - val_accuracy: 0.6667 - val_loss: 0.9183
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.7786 - loss: 0.7664 - val_accuracy: 0.7576 - val_loss: 0.6724
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.9046 - loss: 0.3994 - val_accuracy: 0.8788 - val_loss: 0.5737
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.9618 - loss: 0.2034 - val_accuracy: 0.9242 - val_loss: 0.2225
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.9885 - loss: 0.0765 - val_accuracy: 1.0000 - v

In [ ]:
X_complex_pad = pad_sequences(X_complex_seq, maxlen=40)

In [ ]:
loss, acc = model_dl.evaluate(X_test_pad, y_test_dl)
print("BiLSTM Clean Accuracy:", acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9697 - loss: 0.0718
BiLSTM Clean Accuracy: 0.9696969985961914


In [ ]:
X_complex_seq = tokenizer_dl.texts_to_sequences(complex_df['text'])
X_complex_pad = pad_sequences(X_complex_seq, maxlen=20)

y_complex_dl = le.transform(complex_df['intent'])

loss, acc = model_dl.evaluate(X_complex_pad, y_complex_dl)
print("BiLSTM Complex Accuracy:", acc)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1000 - loss: 3.8158
BiLSTM Complex Accuracy: 0.10000000149011612


In [ ]:
weights = {
    "svm": 0.2,
    "bilstm": 0.3,
    "bert": 0.5
}

In [ ]:
def svm_predict(text):
    vec = vectorizer.transform([text])
    probs = svm_ref.predict_proba(vec)[0]

    pred = svm_ref.classes_[probs.argmax()]
    return pred, float(max(probs))

In [ ]:
def bilstm_predict(text):
    seq = tokenizer_dl.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=40)

    probs = model_dl.predict(pad)[0]
    pred = le.classes_[probs.argmax()]

    return pred, max(probs)

In [ ]:
import torch
import numpy as np

def bert_predict(text):
    bert_ref.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = bert_ref(**inputs)

    probs = torch.softmax(outputs.logits, dim=1).numpy()[0]

    pred = id2label[np.argmax(probs)]

    return pred, float(np.max(probs))

In [ ]:
def hybrid_predict(text):

    svm_pred, svm_conf = svm_predict(text)
    bilstm_pred, bilstm_conf = bilstm_predict(text)
    bert_pred, bert_conf = bert_predict(text)

    scores = {}

    # Add weighted scores
    for pred, conf, w in [
        (svm_pred, svm_conf, weights['svm']),
        (bilstm_pred, bilstm_conf, weights['bilstm']),
        (bert_pred, bert_conf, weights['bert'])
    ]:
        if pred not in scores:
            scores[pred] = 0
        scores[pred] += conf * w

    # Final decision
    final_pred = max(scores, key=scores.get)

    return final_pred, scores

In [ ]:
correct = 0

for i in range(len(complex_df)):
    text = complex_df['text'].iloc[i]
    true_label = complex_df['intent'].iloc[i]

    pred, _ = hybrid_predict(text)

    if pred == true_label:
        correct += 1

hybrid_accuracy = correct / len(complex_df)

print("Hybrid Complex Accuracy:", hybrid_accuracy)

AttributeError: 'BertForSequenceClassification' object has no attribute 'predict_proba'

In [ ]:
weights = {
    "svm": 0.4,
    "bilstm": 0.1,
    "bert": 0.5
}

In [ ]:
def svm_predict(text):
    vec = vectorizer.transform([text])

    # find correct sklearn model automatically
    from sklearn.linear_model import LogisticRegression

    if isinstance(model, LogisticRegression):
        probs = model.predict_proba(vec)[0]
        pred = model.classes_[probs.argmax()]
        return pred, float(max(probs))

    else:
        # fallback: use already trained svm_model if exists
        probs = svm_model.predict_proba(vec)[0]
        pred = svm_model.classes_[probs.argmax()]
        return pred, float(max(probs))

In [ ]:
def bilstm_predict_safe(text):
    seq = tokenizer_dl.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=40)

    probs = model_dl.predict(pad, verbose=0)[0]

    pred = le.classes_[probs.argmax()]
    return pred, float(max(probs))

In [ ]:
def bert_predict(text):
    import torch
    import numpy as np

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)   # <-- uses current BERT model

    probs = torch.softmax(outputs.logits, dim=1).numpy()[0]

    pred = id2label[np.argmax(probs)]

    return pred, float(np.max(probs))

In [ ]:
def hybrid_predict(text):

    svm_pred, svm_conf = svm_predict(text)
    bert_pred, bert_conf = bert_predict(text)

    # Strong confidence threshold
    if bert_conf > 0.75:
        return bert_pred

    # fallback to SVM
    return svm_pred

In [ ]:
correct = 0

for i in range(len(complex_df)):
    text = complex_df['text'].iloc[i]
    true_label = complex_df['intent'].iloc[i]

    pred = hybrid_predict(text)

    if pred == true_label:
        correct += 1

print("Hybrid Accuracy:", correct / len(complex_df))

Hybrid Accuracy: 0.436


In [ ]:
df.to_csv("clean_dataset.csv", index=False)
complex_df.to_csv("complex_dataset.csv", index=False)

In [ ]:
from sklearn.linear_model import LogisticRegression

svm_model = LogisticRegression()
svm_model.fit(X_train_vec, y_train)

LogisticRegression()

In [ ]:
def svm_predict(text):
    vec = vectorizer.transform([text])
    probs = svm_model.predict_proba(vec)[0]

    pred = svm_model.classes_[probs.argmax()]
    return pred, float(max(probs))

In [ ]:
def hybrid_predict(text):

    svm_pred, svm_conf = svm_predict(text)
    bilstm_pred, bilstm_conf = bilstm_predict(text)
    bert_pred, bert_conf = bert_predict(text)

    scores = {}

    for pred, conf, w in [
        (svm_pred, svm_conf, 0.2),
        (bilstm_pred, bilstm_conf, 0.3),
        (bert_pred, bert_conf, 0.5)
    ]:
        scores[pred] = scores.get(pred, 0) + conf * w

    return max(scores, key=scores.get)

In [ ]:
correct = 0

for i in range(len(complex_df)):
    text = complex_df['text'].iloc[i]
    true_label = complex_df['intent'].iloc[i]

    pred = hybrid_predict(text)

    if pred == true_label:
        correct += 1

print("Hybrid Accuracy:", correct / len(complex_df))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 869ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━

In [ ]:
meta_X = []
meta_y = []

for i in range(len(X_test)):
    text = X_test.iloc[i]

    svm_pred, svm_conf = svm_predict(text)
    bert_pred, bert_conf = bert_predict(text)

    meta_X.append([svm_conf, bert_conf])
    meta_y.append(y_test.iloc[i])

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression()
meta_model.fit(meta_X, meta_y)

LogisticRegression()

In [ ]:
def hybrid_predict(text):

    svm_pred, svm_conf = svm_predict(text)
    bert_pred, bert_conf = bert_predict(text)

    meta_features = [[svm_conf, bert_conf]]

    return meta_model.predict(meta_features)[0]

In [ ]:
correct = 0

for i in range(len(complex_df)):
    text = complex_df['text'].iloc[i]
    true_label = complex_df['intent'].iloc[i]

    pred = hybrid_predict(text)

    if pred == true_label:
        correct += 1

print("Hybrid Accuracy:", correct / len(complex_df))

Hybrid Accuracy: 0.1


In [ ]:
def analyze_query(text):
    words = text.split()

    length = len(words)

    # Hinglish detection (simple)
    hinglish_keywords = ["kaise", "kya", "mera", "hai", "karna"]
    hinglish_score = sum(1 for w in words if w.lower() in hinglish_keywords)

    return {
        "length": length,
        "hinglish": hinglish_score > 0
    }

In [ ]:
def hybrid_predict(text):

    features = analyze_query(text)

    svm_pred, svm_conf = svm_predict(text)
    bert_pred, bert_conf = bert_predict(text)
    bilstm_pred, bilstm_conf = bilstm_predict(text)

    # 🔥 RULE 1: Hinglish → avoid BERT
    if features["hinglish"]:
        return svm_pred

    # 🔥 RULE 2: Complex sentence → trust BERT
    if features["length"] > 6:
        return bert_pred

    # 🔥 RULE 3: If models disagree → use confidence
    if len(set([svm_pred, bert_pred, bilstm_pred])) > 1:
        confidences = {
            svm_pred: svm_conf,
            bert_pred: bert_conf,
            bilstm_pred: bilstm_conf
        }
        return max(confidences, key=confidences.get)

    # 🔥 RULE 4: default
    return svm_pred

In [ ]:
correct = 0

for i in range(len(complex_df)):
    text = complex_df['text'].iloc[i]
    true_label = complex_df['intent'].iloc[i]

    pred = hybrid_predict(text)

    if pred == true_label:
        correct += 1

print("Hybrid Accuracy:", correct / len(complex_df))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step
1/1 ━━━